# 06. Inference

Fit lại 5 model trên toàn bộ 136,992 dòng train và dự đoán 34,080 dòng test.

| Cột | Ý nghĩa |
|---|---|
| `model{i}_cash_sign_30d`, `model{i}_card_sign_30d` | Xác suất của từng model |
| `cash_sign_30d`, `card_sign_30d` | Trung bình xác suất của 5 model |
| `uplift_cash_score`, `uplift_card_score` | Uplift ensemble |
| `recommended_strategy` | `Cash` nếu `uplift_cash_score - uplift_card_score >= 0`, ngược lại `Card` |

Xác suất của UPLIFT_RF_CIT là `clip(control_rate + uplift, 0, 1)`, của INT_RF là `P(y=1 | x, W=event)`.

Tập test không có nhãn nên bảng Qini bên dưới là Qini trên chính tập train.

In [ ]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
from src import config as C
from src.ensemble import fit_predict_ensemble
from src.explain import train_qini_table
from src.preprocessing import load_model_data, load_test_matrix

X, W, y_cash, y_card, feat_cols = load_model_data()
X_test = load_test_matrix(feat_cols)
X.shape, X_test.shape

In [ ]:
pred, models = fit_predict_ensemble(X, W, y_cash, y_card, X_test)

In [ ]:
train_qini = train_qini_table(models, X, W, y_cash, y_card)
train_qini.round(6)

In [ ]:
submission = pd.DataFrame({"ID": np.arange(len(X_test))})
for i, name in enumerate(models, 1):
    submission[f"model{i}_cash_sign_30d"] = pred["cash"]["prob"][name]
    submission[f"model{i}_card_sign_30d"] = pred["card"]["prob"][name]
submission[C.TARGET_CASH] = pred["cash"]["prob_mean"]
submission[C.TARGET_CARD] = pred["card"]["prob_mean"]
submission["uplift_cash_score"] = pred["cash"]["uplift_mean"]
submission["uplift_card_score"] = pred["card"]["uplift_mean"]
submission["recommended_strategy"] = np.where(submission["uplift_cash_score"] - submission["uplift_card_score"] >= 0, "Cash", "Card")
submission.head()

In [ ]:
submission["recommended_strategy"].value_counts()

In [ ]:
out = C.RESULTS_DIR / "inference"
out.mkdir(exist_ok=True)
submission.to_csv(out / "submission.csv", index=False)
submission[C.TARGETS].to_csv(out / "test_predictions.csv", index=False)
train_qini.to_csv(out / "train_qini_full_fit.csv", index=False)
joblib.dump({"models": models, "feat_cols": feat_cols}, out / "ensemble_models.joblib")